# Demo v0.3 — парковочные сценарии

Три типа машино-мест, каждый со своей физикой использования квартала:

| Тип | Площадь поверхности | Норматив |
|---|---|---|
| **Открытые** | 20.75 м²/м.м. | в составе ЗУ жилья |
| **Многоуровневые** (наземные) | 10–30 м²/м.м. в зависимости от этажности | отдельный балансовый компонент; макс. 300 м/м на объект (СПб) |
| **Подземные** | 0 м² | поверхностной нагрузки нет |

Минимум открытых м/м фиксирован ПЗЗ СПб — **12.5%** от общего числа требуемых.

**`ParkingConfig.mode`** — три режима:
- `min_open` — открытых ровно минимум, остальное в подземных (по умолчанию)
- `all_open` — всё открытое; «дешёвый» сценарий с максимальной поверхностной нагрузкой
- `custom` — пользователь задаёт доли каждого типа

In [1]:
import pathlib
import pandas as pd

from urban_model import compare_scenarios
from urban_model.models import Site, CalculationOptions, ParkingConfig, Scenario
from urban_model.normatives import load_normatives
from urban_model.modes.compare import run_scenarios
from urban_model.export import to_xlsx

norms = load_normatives('spb')

## Три парковочных сценария на одном квартале (10 га)

Один и тот же квартал, одна этажность жилья (15 этажей), одно ППТ — отличается только парковочный сценарий.

In [2]:
site = Site(area_m2=100_000, name='Квартал 10 га')

scenarios = [
    Scenario(
        name='Min open + подземные',
        site=site,
        options=CalculationOptions(
            floors=15, planning_doc=True,
            parking=ParkingConfig(mode='min_open'),
        ),
    ),
    Scenario(
        name='100% открытые',
        site=site,
        options=CalculationOptions(
            floors=15, planning_doc=True,
            parking=ParkingConfig(mode='all_open'),
        ),
    ),
    Scenario(
        name='15% откр + 50% МП(4 ур.) + 35% подз',
        site=site,
        options=CalculationOptions(
            floors=15, planning_doc=True,
            parking=ParkingConfig(
                mode='custom',
                open_share=0.15,
                multilevel_share=0.50,
                underground_share=0.35,
                multilevel_levels=4,
            ),
        ),
    ),
]

df = compare_scenarios(scenarios, norms)
# Сравнение «по парковкам и балансу»
rows_of_interest = [
    'КИТ',
    'Площадь квартир, м²',
    'Население, чел',
    'Парковки всего, м/м',
    'Откр. парковки, м/м',
    'Откр. парковки, м²',
    'Многоуровн. парковки, м/м',
    'Многоуровн. паркинги, шт',
    'Многоуровн. паркинги, м²',
    'Подземные парковки, м/м',
    'Баланс территории, м²',
    'Ограничивающий фактор',
]
df.loc[rows_of_interest]

сценарий,Min open + подземные,100% открытые,15% откр + 50% МП(4 ур.) + 35% подз
показатель,,,
КИТ,1.2,1.14,1.2
"Площадь квартир, м²",89985.35,85502.93,89985.35
"Население, чел",3213.76,3053.68,3213.76
"Парковки всего, м/м",1125,1069,1125
"Откр. парковки, м/м",141,1069,169
"Откр. парковки, м²",2925.75,22181.75,3506.75
"Многоуровн. парковки, м/м",0,0,562
"Многоуровн. паркинги, шт",0,0,2
"Многоуровн. паркинги, м²",0.0,0.0,6744.0


### Что видно из сравнения

1. **`min_open`** даёт максимальный КИТ — поверхность не съедается парковкой, остаток уходит в подземные.
2. **`all_open`** — самый «честный» по нагрузке сценарий. Открытые парковки занимают ~22 тыс. м² поверхности, поэтому максимально допустимый КИТ ниже. Видно сразу, какой ценой оплачивается «дешёвая» парковка.
3. **`custom`** позволяет посмотреть промежуточный вариант — например, замена части подземных на многоуровневые наземные (4 уровня) даёт рост поверхностной нагрузки, но не такой сильный, как при 100% открытых.

## Детальная сводка по последнему сценарию

In [3]:
pairs = run_scenarios(scenarios, norms)
for name, res in pairs:
    print(f'═══ {name} ═══')
    print(res.summary())
    print()

═══ Min open + подземные ═══
Профиль: spb
КИТ:                     1.200 (норм. макс 2.5)
Площадь квартир:         89,985 м²
Население:               3,214 чел
Плотность:               321.4 чел/га [ok]
ДОО (мест):              требуется 196.03951590401783 → принято 200
СОШ (мест):              требуется 385.6515066964285 → принято 390
ЗНОП (м²/чел):           0 → итого 0 м²
Парковки (м/м):
  всего требуется        1125
  открытые               141 м/м, 2,926 м²
  подземные              984 м/м (без поверхностной площади)
Баланс:                  OK (+16,180 м²)
Ограничивающий фактор:   ЗУ жилой застройки (41,220 м², 41.2% квартала)
  ⚠ СОШ: расчётная вместимость [390] < нормативного минимума 550 мест — стандартная отдельно стоящая СОШ невозможна, нужна стоянка-спутник или ВПП-школа (учтётся в v0.2).

═══ 100% открытые ═══
Профиль: spb
КИТ:                     1.140 (норм. макс 2.5)
Площадь квартир:         85,503 м²
Население:               3,054 чел
Плотность:               305.4 чел

## Предупреждение о малых СОШ

Если на квартале мало населения, расчёт даёт СОШ ниже нормативного минимума 550 мест. Это **не ошибка** — это `WARNING` со ссылкой: «нужна стоянка-спутник или ВПП-школа» (учтётся в v0.2).

In [4]:
from urban_model.modes.verify import verify_kit

small = verify_kit(0.5, Site(area_m2=20_000), CalculationOptions(floors=8), norms)
print(f'СОШ принято:  {small.school_places_accepted.value} мест')
print(f'Статус поля:  {small.school_places_accepted.status.value}')
print()
for w in small.warnings:
    print(f'⚠ {w}')

СОШ принято:  40 мест
Статус поля:  warning

⚠ СОШ: расчётная вместимость [40] < нормативного минимума 550 мест — стандартная отдельно стоящая СОШ невозможна, нужна стоянка-спутник или ВПП-школа (учтётся в v0.2).


## Экспорт в Excel

Файл `output/tep_parking_scenarios.xlsx`:
- лист «Сравнение» — сводная таблица КПЭ
- лист «Аудит» — все поля × сценарии × источники

Цветом помечены: `ok` зелёный, `warning` жёлтый, `error`/`ДЕФИЦИТ` красный.

In [5]:
out_dir = pathlib.Path('../output')
out_dir.mkdir(exist_ok=True)
path = to_xlsx(pairs, out_dir / 'tep_parking_scenarios.xlsx')
print(f'Записан: {path.resolve()}')
print(f'Размер:  {path.stat().st_size:,} байт')

Записан: D:\Github\my_urban_model\output\tep_parking_scenarios.xlsx
Размер:  11,495 байт
